# Phase 2 — B0 Sequential No-Replay (Kaggle)

Runs B0 for the frozen five-language XNLI protocol using the Phase 1 candidate-label prompt and shared LoRA trainer. By default it runs all five fixed stream orders (one training seed each), evaluates every language before training and after each stage, saves stage adapters plus result ledgers, and writes a validated result JSON per completed order.

## Before running
1. Enable a Kaggle GPU and Internet.
2. Accept the gated model terms for `meta-llama/Llama-3.2-1B` and add a Kaggle secret named `HF_TOKEN`.
3. This GitHub repository is public, so no GitHub token/API key is needed to clone it. The URL currently redirects to a repository name ending in a period; keep the exact `REPO_URL` below.
4. Add optional `WANDB_API_KEY` only if you enable W&B.
5. Run all cells, then use **Save Version** to persist `/kaggle/working/phase2_artifacts` as notebook output. A run interrupted after a stage leaves its checkpoint/ledger; inspect it and use a fresh artifact root to restart that incomplete run.

This notebook runs the complete five-order B0 protocol; it does not offer a partial-run mode. Completed order results are validated and skipped on rerun, so rerunning safely resumes only missing orders. At the end it verifies all five result records and writes `B0_summary.json`.

In [1]:
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = "https://github.com/Mani-S-49/asymmetric-interference-continual-learning..git"
REPO_ROOT = Path("/kaggle/working/asymmetric-interference-continual-learning")
if not REPO_ROOT.exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_ROOT)], check=True)
elif (REPO_ROOT / ".git").exists():
    subprocess.run(["git", "-C", str(REPO_ROOT), "remote", "set-url", "origin", REPO_URL], check=True)
    subprocess.run(["git", "-C", str(REPO_ROOT), "pull", "--ff-only"], check=True)
if not (REPO_ROOT / "phase2" / "run_baseline.py").is_file():
    raise FileNotFoundError("The cloned repository is missing the Phase 2 B0/B1 runner. Push the current repository changes, then restart this notebook.")

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO_ROOT / "phase2" / "requirements-phase2.txt")], check=True)
sys.path.insert(0, str(REPO_ROOT))

from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()
hf_token = secrets.get_secret("HF_TOKEN")
if not hf_token:
    raise RuntimeError("HF_TOKEN Kaggle secret is empty.")
os.environ["HF_TOKEN"] = hf_token
del hf_token

USE_WANDB = False
WANDB_PROJECT = "phase2-xnli-baselines"
if USE_WANDB:
    wandb_key = secrets.get_secret("WANDB_API_KEY")
    if not wandb_key:
        raise RuntimeError("Enable USE_WANDB only after adding the WANDB_API_KEY Kaggle secret.")
    os.environ["WANDB_API_KEY"] = wandb_key
    del wandb_key

PROTOCOL_PATH = REPO_ROOT / "phase2" / "configs" / "pilot_protocol.json"
ARTIFACT_ROOT = Path("/kaggle/working/phase2_artifacts")
print({"repo": str(REPO_ROOT), "artifact_root": str(ARTIFACT_ROOT), "wandb": USE_WANDB})

Cloning into '/kaggle/working/asymmetric-interference-continual-learning'...


{'repo': '/kaggle/working/asymmetric-interference-continual-learning', 'artifact_root': '/kaggle/working/phase2_artifacts', 'wandb': False}


In [2]:
from phase2.preflight import collect_preflight
from phase2.prepare_data import prepare_xnli
from phase2.protocol import build_manifest, load_protocol
from phase2.results import validate_result_record
from phase2.run_baseline import validate_prepared_artifacts
import json

preflight = collect_preflight(require_cuda=True, check_gated_model=True)
print(json.dumps(preflight, indent=2, sort_keys=True))

protocol = load_protocol(PROTOCOL_PATH)
if not (ARTIFACT_ROOT / "protocol_manifest.json").is_file():
    if (ARTIFACT_ROOT / "data").exists():
        raise FileExistsError("A partial data directory exists without its manifest. Inspect it before using a fresh artifact root.")
    prepared_manifest = prepare_xnli(PROTOCOL_PATH, ARTIFACT_ROOT)
    print("Prepared fixed XNLI subsets:", prepared_manifest["expected_run_count"], "scheduled runs")
else:
    prepared_manifest = validate_prepared_artifacts(protocol, ARTIFACT_ROOT)
    print("Verified existing fixed data manifest:", prepared_manifest["protocol_sha256"])

ORDER_IDS = [order["id"] for order in protocol["orders"]]
if len(ORDER_IDS) != 5:
    raise ValueError(f"B0 requires all five protocol orders; found {len(ORDER_IDS)}.")
print({"languages": protocol["dataset"]["languages"], "orders_to_run": ORDER_IDS, "buffer_capacity": protocol["replay"]["buffer_capacity_examples"], "data_manifest_sha256": prepared_manifest["data_manifest_sha256"]})

config.json:   0%|          | 0.00/843 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/50.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/301 [00:00<?, ?B/s]

{
  "cuda_available": true,
  "cuda_device": "Tesla T4",
  "cuda_device_count": 2,
  "cuda_total_memory_gib": 14.56,
  "gated_model_metadata_checked": true,
  "huggingface_token_present": true,
  "model_id": "meta-llama/Llama-3.2-1B",
  "package_versions": {
    "accelerate": "1.15.0",
    "bitsandbytes": "0.50.2",
    "datasets": "5.0.1",
    "peft": "0.21.2",
    "torch": "2.11.0+cu130",
    "transformers": "5.18.0"
  },
  "python_version": "3.13.16",
  "tokenizer_pad_token_present": false
}


README.md:   0%|          | 0.00/20.8k [00:00<?, ?B/s]

... (42 outputs left out)

Generating train split:   0%|          | 0/392702 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/5010 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2490 [00:00<?, ? examples/s]

Prepared fixed XNLI subsets: 41 scheduled runs
{'languages': ['en', 'hi', 'es', 'de', 'sw'], 'orders_to_run': ['order-1', 'order-2', 'order-3', 'order-4', 'order-5'], 'buffer_capacity': 125, 'data_manifest_sha256': 'e459305688bf0ff5db4f744cee773ef2c0bb55c52df4bf71a7980479e58301a4'}


In [3]:
import subprocess
import sys

command = [sys.executable, "-m", "phase2.run_baseline", "--method", "B0", "--protocol", str(PROTOCOL_PATH), "--artifact-root", str(ARTIFACT_ROOT)]
for order_id in ORDER_IDS:
    command.extend(["--order-id", order_id])
if USE_WANDB:
    command.extend(["--wandb-project", WANDB_PROJECT])
subprocess.run(command, cwd=REPO_ROOT, check=True)

run_manifest = build_manifest(protocol)
run_manifest["data_artifacts"] = prepared_manifest["data_artifacts"]
run_manifest["data_manifest_sha256"] = prepared_manifest["data_manifest_sha256"]
expected_runs = {run["run_id"]: run for run in run_manifest["runs"] if run["method_id"] == "B0"}
if len(expected_runs) != 5:
    raise ValueError(f"Expected five B0 runs in the current manifest; found {len(expected_runs)}.")
completed = {}
for run_id, run in expected_runs.items():
    result_path = ARTIFACT_ROOT / "results" / f"{run_id}.json"
    if not result_path.is_file():
        raise FileNotFoundError(f"B0 order is incomplete: {result_path}")
    record = json.loads(result_path.read_text(encoding="utf-8"))
    validate_result_record(record, run_manifest)
    completed[run["order_id"]] = record

import statistics
metric_names = ("ACC", "BWT", "FWT", "F")
summary = {
    "method_id": "B0",
    "protocol_id": run_manifest["protocol_id"],
    "protocol_sha256": run_manifest["protocol_sha256"],
    "data_manifest_sha256": run_manifest["data_manifest_sha256"],
    "completed_orders": len(completed),
    "expected_orders": len(expected_runs),
    "per_order": {order_id: {name: record["metrics"][name] for name in metric_names} for order_id, record in sorted(completed.items())},
    "across_order_summary": {
        name: {
            "mean": statistics.mean(record["metrics"][name] for record in completed.values()),
            "population_std": statistics.pstdev(record["metrics"][name] for record in completed.values()),
        }
        for name in metric_names
    },
}
summary_path = ARTIFACT_ROOT / "B0_summary.json"
temporary_path = summary_path.with_suffix(".json.tmp")
temporary_path.write_text(json.dumps(summary, indent=2, sort_keys=True) + "\n", encoding="utf-8")
temporary_path.replace(summary_path)
print(json.dumps(summary, indent=2, sort_keys=True))
print("Validated all five B0 orders. Summary:", summary_path)

Starting xnli-pilot-5lang-500x200-v1-B0-order-1-seed-42


Tokenizing causal-LM prompt/target data: 100%|██████████| 500/500 [00:00<00:00, 3590.20 examples/s]
[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 128001}.
Tokenizing causal-LM prompt/target data:   0%|          | 0/500 [00:00<?, ? examples/s]

{'loss': '1.575', 'grad_norm': '9.417', 'learning_rate': '0.0001437', 'epoch': '0.32'}
{'loss': '0.9898', 'grad_norm': '6.61', 'learning_rate': '8.125e-05', 'epoch': '0.64'}
{'loss': '0.9255', 'grad_norm': '3.072', 'learning_rate': '1.875e-05', 'epoch': '0.96'}
{'train_runtime': '136.2', 'train_samples_per_second': '3.671', 'train_steps_per_second': '0.235', 'train_loss': '1.153', 'epoch': '1'}


100%|██████████| 32/32 [04:04<00:00,  7.64s/it]


{'loss': '1.133', 'grad_norm': '3.436', 'learning_rate': '0.0001437', 'epoch': '0.32'}
{'loss': '1.006', 'grad_norm': '7.399', 'learning_rate': '8.125e-05', 'epoch': '0.64'}
{'loss': '0.9332', 'grad_norm': '1.814', 'learning_rate': '1.875e-05', 'epoch': '0.96'}
{'train_runtime': '244.5', 'train_samples_per_second': '2.045', 'train_steps_per_second': '0.131', 'train_loss': '1.012', 'epoch': '1'}


100%|██████████| 32/32 [02:35<00:00,  4.86s/it]


... (41 outputs left out)

100%|██████████| 32/32 [02:16<00:00,  4.25s/it]


{'loss': '0.9644', 'grad_norm': '6.015', 'learning_rate': '0.0001437', 'epoch': '0.32'}
{'loss': '0.7957', 'grad_norm': '6.758', 'learning_rate': '8.125e-05', 'epoch': '0.64'}
{'loss': '0.7193', 'grad_norm': '7.714', 'learning_rate': '1.875e-05', 'epoch': '0.96'}
{'train_runtime': '136', 'train_samples_per_second': '3.675', 'train_steps_per_second': '0.235', 'train_loss': '0.8295', 'epoch': '1'}


100%|██████████| 32/32 [02:35<00:00,  4.86s/it]


{'loss': '0.7745', 'grad_norm': '6.116', 'learning_rate': '0.0001437', 'epoch': '0.32'}
{'loss': '0.703', 'grad_norm': '6.626', 'learning_rate': '8.125e-05', 'epoch': '0.64'}
{'loss': '0.6786', 'grad_norm': '4.205', 'learning_rate': '1.875e-05', 'epoch': '0.96'}
{'train_runtime': '155.6', 'train_samples_per_second': '3.214', 'train_steps_per_second': '0.206', 'train_loss': '0.7099', 'epoch': '1'}
Completed xnli-pilot-5lang-500x200-v1-B0-order-5-seed-42: ACC=0.6490
{
  "across_order_summary": {
    "ACC": {
      "mean": 0.5822,
      "population_std": 0.04837933443113909
    },
    "BWT": {
      "mean": 0.09825,
      "population_std": 0.044690603039117746
    },
    "F": {
      "mean": -0.04575,
      "population_std": 0.04745129081489775
    },
    "FWT": {
      "mean": 0.11000000000000001,
      "population_std": 0.02738612787525831
    }
  },
  "completed_orders": 5,
  "data_manifest_sha256": "e459305688bf0ff5db4f744cee773ef2c0bb55c52df4bf71a7980479e58301a4",
  "expected_orders"

## Saved outputs

- `phase2_artifacts/protocol_manifest.json` and `data/`: frozen selected examples and checksums.
- `phase2_artifacts/checkpoints/<run_id>/`: adapter/tokenizer after each language stage.
- `phase2_artifacts/ledgers/<run_id>/`: complete stage-by-language evaluation matrix and CSV ledger.
- `phase2_artifacts/results/<run_id>.json`: validated Phase 2 result record for aggregation.
- `phase2_artifacts/B0_summary.json`: summary of all five validated orders and across-order metrics.

The B0 `BWT`, `FWT`, and forgetting metrics are computed from the complete fixed-test performance matrix. Do not select or tune settings using the test scores.